# DATASCI 207 Final Project — XGBoost Volatility Baseline (Walk-Forward, Full Coverage After Warm-Up)

**Author:** Vinit S. Bhatt  
**Date:** 2026-03-03  
**Environment:** Google Colab  
**Input location:** `My Drive / Colab Notebooks /`  

---

## What this notebook does (one and only one model approach)

This notebook implements **one** XGBoost baseline approach:

### ✅ Walk-forward expanding-window XGBoost (per ticker), with full coverage after warm-up

For each ETF:
- We wait for a **warm-up** period (enough history to build lag features and to train a model).
- Then we generate baseline predictions for **every remaining row/date**, in **contiguous blocks** (no fold-boundary gaps).

You can run it in two modes:
- **Single ETF mode:** set `SELECTED_TICKER = "SPY"` (fast for debugging)
- **All ETF mode:** set `SELECTED_TICKER = None` (runs all tickers)

The notebook outputs a new column:

**Setup note:** This notebook can prompt you to adjust warm-up/retraining settings (and explains pros/cons) before it runs.


- `baseline_xgb_walkforward`

**Important:** Any missing baseline values should occur only:
- at the beginning (warm-up), or
- where the dataset itself has missing rows/fields.

---

## No-leakage guarantee (two gates)

1) **Feature gate:** lag/trailing features use only information available at or before time *t*  
2) **Training gate:** for each prediction block, training uses only rows strictly earlier than the prediction block

If either gate is violated, the baseline is invalid. This notebook keeps both gates closed.


In [1]:
# Purpose:
#   Ensure required packages exist in this Colab runtime.
# Steps:
#   1) Install or upgrade xgboost and scikit-learn.
#   2) Keep output quiet to reduce noise.

!pip -q install -U xgboost scikit-learn

print("Package install complete.")


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 25.4 MB/s eta 0:00:00
Package install complete.


In [2]:
# Purpose:
#   Mount Google Drive so Colab can read and write files under MyDrive.

from google.colab import drive
drive.mount("/content/drive")

print("Google Drive mounted at /content/drive/MyDrive/")


Mounted at /content/drive
Google Drive mounted at /content/drive/MyDrive/


In [3]:
# Python 3.9 compatibility: makes all type hints lazy strings
from __future__ import annotations

# Purpose:
#   Import libraries for data handling, modeling, and evaluation.

import os
import math
import inspect
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

from xgboost import XGBRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

from IPython.display import display

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 160)


In [4]:
# Purpose:
#   Configure file paths, dataset schema, run mode (single ticker vs all tickers),
#   and walk-forward settings.

# -----------------------------
# Drive folder (input + output)
# -----------------------------
DRIVE_FOLDER = "/content/drive/MyDrive/Colab Notebooks/"

# -----------------------------
# Input CSV selection
# -----------------------------
# Option A (recommended): set the exact filename.
INPUT_CSV_FILENAME = "vol_dataset_with_4_baselines_022326.csv"   # <-- change if needed

# Option B: auto-find the first CSV in DRIVE_FOLDER if you set INPUT_CSV_FILENAME to "YOUR_FILE.csv"
AUTO_FIND_CSV = True

def find_first_csv(folder: str) -> str | None:
    try:
        candidates = sorted([f for f in os.listdir(folder) if f.lower().endswith(".csv")])
        return os.path.join(folder, candidates[0]) if candidates else None
    except FileNotFoundError:
        return None

if INPUT_CSV_FILENAME and INPUT_CSV_FILENAME != "YOUR_FILE.csv":
    DATA_PATH = os.path.join(DRIVE_FOLDER, INPUT_CSV_FILENAME)
elif AUTO_FIND_CSV:
    DATA_PATH = find_first_csv(DRIVE_FOLDER)
    print(f"Auto-found CSV: {DATA_PATH}")
else:
    raise ValueError("Set INPUT_CSV_FILENAME or enable AUTO_FIND_CSV.")

# -----------------------------
# Column names (match dataset)
# -----------------------------
DATE_COL   = "date"
TICKER_COL = "ticker"
TARGET_COL = "forward_vol_5d_annual_decimel_calculated"

# Existing rolling baselines (comparison only; NOT used as features by default)
ROLLING_BASELINE_COLS = ["baseline_avg_5", "baseline_avg_20", "baseline_ols_5", "baseline_ols_20"]

# -----------------------------
# Run mode: single ETF or all ETFs
# -----------------------------
# Set SELECTED_TICKER to a ticker symbol like "SPY" to run only one ETF.
# Set SELECTED_TICKER = None to run all tickers in the dataset.
SELECTED_TICKER = None   # e.g., "SPY"

# If True and SELECTED_TICKER is set, the output CSV will contain ONLY that ticker's rows.
OUTPUT_ONLY_SELECTED_TICKER = True

# -----------------------------
# Feature design toggle
# -----------------------------
# Important: Using rolling baseline columns as features can be interpreted as "stacking"
# (the model learns from other baselines you also compare against).
# The cleanest baseline story is to keep this False.
USE_ROLLING_BASELINES_AS_FEATURES = False

# -----------------------------
# Walk-forward configuration (FULL COVERAGE AFTER WARM-UP)
# -----------------------------
# Goal: Fill baseline values for *every* row/date after a per-ticker warm-up period.
#
# Warm-up rules (per ticker):
#   - Must have enough past rows to compute lag features (max lag)
#   - Must have enough past NON-NULL target rows to train a model
#
# After warm-up, we predict in contiguous blocks of WF_STEP_ROWS rows.
# A good default is ~252 rows (~1 trading year). Larger blocks retrain less often (faster).
#
# IMPORTANT:
#   This notebook can PROMPT you to adjust these settings. Prompts are useful when you
#   want to trade off (A) earlier baseline coverage vs (B) stability vs (C) runtime.
#
# If you prefer zero prompts (for "Run all" convenience), set INTERACTIVE_SETUP = False.
INTERACTIVE_SETUP = False  # Set to True ONLY if running cells one-by-one interactively.
                           # IMPORTANT: Keep False for "Run all" — input() blocks in Colab
                           # when cells are run programmatically and raises StdinNotImplementedError.

# ---------
# Defaults
# ---------
MAX_LAG_DAYS_DEFAULT     = 20
MIN_TRAIN_ROWS_Y_DEFAULT = 125
WF_STEP_ROWS_DEFAULT     = 252
WF_EARLY_STOP_SPLIT_DEFAULT = 0.85  # first 85% fit, last 15% eval (still before prediction block)

def _prompt_int(name: str, default: int, explanation: str, min_val: int | None = None) -> int:
    print("\n" + "="*80)
    print(f"{name}")
    print(explanation)
    if min_val is not None:
        print(f"Minimum allowed: {min_val}")
    raw = input(f"Enter {name} (press Enter to keep default {default}): ").strip()
    if raw == "":
        return default
    try:
        val = int(raw)
    except ValueError:
        print(f"Invalid input. Keeping default {default}.")
        return default
    if min_val is not None and val < min_val:
        print(f"{name} too small. Using minimum {min_val}.")
        return min_val
    return val

def _prompt_float(name: str, default: float, explanation: str, min_val: float | None = None, max_val: float | None = None) -> float:
    print("\n" + "="*80)
    print(f"{name}")
    print(explanation)
    if (min_val is not None) or (max_val is not None):
        print(f"Allowed range: [{min_val}, {max_val}]")
    raw = input(f"Enter {name} (press Enter to keep default {default}): ").strip()
    if raw == "":
        return default
    try:
        val = float(raw)
    except ValueError:
        print(f"Invalid input. Keeping default {default}.")
        return default
    if min_val is not None and val < min_val:
        print(f"{name} too small. Using {min_val}.")
        val = min_val
    if max_val is not None and val > max_val:
        print(f"{name} too large. Using {max_val}.")
        val = max_val
    return val

# -----------------------------
# Choose settings (interactive)
# -----------------------------
# NOTE on consequences:
#   MIN_TRAIN_ROWS_Y (warm-up length):
#     - Higher (e.g., 500): baseline starts later, but early predictions are more stable.
#     - Medium (e.g., 250): good balance (about ~1 trading year).
#     - Lower (e.g., 125): baseline starts earlier, but predictions can be noisier / higher overfit risk.
#
#   WF_STEP_ROWS (retrain frequency):
#     - Smaller (e.g., 126): retrain more often; more adaptive; slower runtime.
#     - Default (252): reasonable tradeoff.
#     - Larger (504+): much faster; less adaptive to regime changes.
#
#   MAX_LAG_DAYS:
#     - Must be >= the largest lag you create in feature engineering.
#       This notebook uses lag days [1, 5, 20], so MAX_LAG_DAYS must be >= 20 unless you also change LAG_DAYS.
#
if INTERACTIVE_SETUP:
    MAX_LAG_DAYS = _prompt_int(
        name="MAX_LAG_DAYS",
        default=MAX_LAG_DAYS_DEFAULT,
        explanation=(
            "Controls warm-up required for lag features.\n"
            "Consequence: larger value delays the first baseline date slightly, but ensures lag features exist."
        ),
        min_val=1,
    )

    # Enforce compatibility with the lag features later in the notebook.
    # Current lag set is [1, 5, 20] so MAX_LAG_DAYS must be at least 20.
    if MAX_LAG_DAYS < 20:
        print("\nNOTE: This notebook creates 20-day lag features. Setting MAX_LAG_DAYS < 20 would be inconsistent.")
        print("Auto-adjusting MAX_LAG_DAYS to 20. If you truly want smaller, also change LAG_DAYS later.")
        MAX_LAG_DAYS = 20

    MIN_TRAIN_ROWS_Y = _prompt_int(
        name="MIN_TRAIN_ROWS_Y",
        default=MIN_TRAIN_ROWS_Y_DEFAULT,
        explanation=(
            "Minimum number of past NON-NULL target rows required before we start predicting (warm-up).\n"
            "Consequence: smaller value = baseline starts earlier, but may be noisier / higher overfit risk."
        ),
        min_val=60,
    )

    WF_STEP_ROWS = _prompt_int(
        name="WF_STEP_ROWS",
        default=WF_STEP_ROWS_DEFAULT,
        explanation=(
            "Prediction block size in rows. We retrain once per block.\n"
            "Consequence: smaller value = more frequent retraining (more compute), larger value = faster but less adaptive."
        ),
        min_val=20,
    )

    WF_EARLY_STOP_SPLIT = _prompt_float(
        name="WF_EARLY_STOP_SPLIT",
        default=WF_EARLY_STOP_SPLIT_DEFAULT,
        explanation=(
            "Fraction of training rows used for model fitting vs early-stopping evaluation.\n"
            "Consequence: larger value leaves fewer rows for early-stop evaluation; smaller value leaves fewer for fitting."
        ),
        min_val=0.7,
        max_val=0.95,
    )
else:
    MAX_LAG_DAYS = MAX_LAG_DAYS_DEFAULT
    MIN_TRAIN_ROWS_Y = MIN_TRAIN_ROWS_Y_DEFAULT
    WF_STEP_ROWS = WF_STEP_ROWS_DEFAULT
    WF_EARLY_STOP_SPLIT = WF_EARLY_STOP_SPLIT_DEFAULT

print("\n" + "="*80)
print("Walk-forward settings in effect:")
print("  MAX_LAG_DAYS      =", MAX_LAG_DAYS)
print("  MIN_TRAIN_ROWS_Y  =", MIN_TRAIN_ROWS_Y)
print("  WF_STEP_ROWS      =", WF_STEP_ROWS)
print("  WF_EARLY_STOP_SPLIT =", WF_EARLY_STOP_SPLIT)
print("="*80)


# -----------------------------
# XGBoost hyperparameters (baseline values)
# -----------------------------
RANDOM_STATE = 42
EARLY_STOPPING_ROUNDS = 50

XGB_PARAMS = dict(
    n_estimators=2000,          # upper bound; early stopping finds best iteration
    learning_rate=0.03,
    max_depth=3,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_lambda=1.0,
    min_child_weight=1.0,
    objective="reg:squarederror",
    eval_metric="rmse",
    random_state=RANDOM_STATE,
    n_jobs=-1,
)

print("DATA_PATH =", DATA_PATH)
print("SELECTED_TICKER =", SELECTED_TICKER)



Walk-forward settings in effect:
  MAX_LAG_DAYS      = 20
  MIN_TRAIN_ROWS_Y  = 125
  WF_STEP_ROWS      = 252
  WF_EARLY_STOP_SPLIT = 0.85
DATA_PATH = /content/drive/MyDrive/Colab Notebooks/vol_dataset_with_4_baselines_022326.csv
SELECTED_TICKER = None


In [5]:
# Purpose:
#   Load the dataset, enforce types, and run sanity checks.

if DATA_PATH is None or (not os.path.exists(DATA_PATH)):
    raise FileNotFoundError(
        f"Could not find the input CSV at: {DATA_PATH}\n"
        "Check INPUT_CSV_FILENAME and that the file exists in MyDrive/Colab Notebooks."
    )

df_raw = pd.read_csv(DATA_PATH, low_memory=False)

# Required schema checks
required = {DATE_COL, TICKER_COL, TARGET_COL}
missing = sorted(list(required - set(df_raw.columns)))
if missing:
    print("Missing required columns:", missing)
    print("Available columns sample:", list(df_raw.columns)[:40])
    raise ValueError("Dataset schema mismatch. Update DATE_COL/TICKER_COL/TARGET_COL accordingly.")

df = df_raw.copy()
df[DATE_COL] = pd.to_datetime(df[DATE_COL])
df = df.sort_values([TICKER_COL, DATE_COL]).reset_index(drop=True)

print("Shape:", df.shape)
print("Tickers:", df[TICKER_COL].nunique())
print("Date range:", df[DATE_COL].min().date(), "to", df[DATE_COL].max().date())
print("Target non-null rate:", round(float(df[TARGET_COL].notna().mean()), 6))

# Determine tickers to run
all_tickers = sorted(df[TICKER_COL].dropna().unique().tolist())
tickers_to_run = [SELECTED_TICKER] if SELECTED_TICKER else all_tickers

missing_sel = [t for t in tickers_to_run if t not in all_tickers]
if missing_sel:
    raise ValueError(f"Selected ticker(s) not found in dataset: {missing_sel}")

print("Tickers to run:", tickers_to_run[:10], ("..." if len(tickers_to_run) > 10 else ""))


Shape: (96525, 28)
Tickers: 36
Date range: 2014-12-31 to 2025-12-31
Target non-null rate: 0.998135
Tickers to run: ['BEDZ', 'FTXG', 'IYZ', 'KBE', 'KCE', 'KIE', 'KRE', 'PEJ', 'PNQI', 'PXQ'] ...


## Feature engineering (leakage-safe)

Design rules:

- Use only values known **at or before** date *t*
- Create lags using `groupby(ticker).shift(L)` so the feature at *t* uses values from *t-L*
- Avoid using the forward target (or anything derived from t+1..t+5) as a feature

This notebook builds:

- Base trailing features (if present)
- Calendar features (from the date)
- HAR-like lags (1, 5, 20 trading days) for a small set of stable predictors

It automatically skips candidate features that do not exist in your dataset.


In [6]:
# Purpose:
#   Build the feature columns list and create lagged features within each ticker.

# Candidate base features (safe if they are trailing / contemporaneous at t)
BASE_FEATURES_CANDIDATE = [
    # The most important safe persistence signal (shifted label; known at time t)
    "y_known_at_t",

    # trailing volatility (FactSet)
    "trailing_vol_annual_decimel_5d_factset",
    "trailing_vol_annual_decimel_20d_factset",

    # trailing volatility (calculated) — multiple naming variants supported
    "trailing_vol__annual_decimel_5d_calculated",
    "trailing_vol__annual_decimel_20d_calculated",
    "trailing_vol_annual_decimel_5d_calculated",
    "trailing_vol_annual_decimel_20d_calculated",

    # macro and risk series (assumed time-aligned at date t)
    "VIX",
    "NYGOLDS",
    "OIL_WTI_S",
    "US_10Y_BOND_YLD",
    "US_3M_TB_YLD",
]

# Optionally include rolling baselines as features (usually keep False for a clean baseline story)
if USE_ROLLING_BASELINES_AS_FEATURES:
    BASE_FEATURES_CANDIDATE = [
        "baseline_avg_5", "baseline_avg_20", "baseline_ols_5", "baseline_ols_20"
    ] + BASE_FEATURES_CANDIDATE

BASE_FEATURES = [c for c in BASE_FEATURES_CANDIDATE if c in df.columns]

# Calendar features (date-derived; always leakage-safe)
df_feat = df.copy()
df_feat["day_of_week"] = df_feat[DATE_COL].dt.dayofweek
df_feat["month"]       = df_feat[DATE_COL].dt.month
df_feat["quarter"]     = df_feat[DATE_COL].dt.quarter
CALENDAR_FEATURES = ["day_of_week", "month", "quarter"]

# Lag sources (subset of stable predictors)
LAG_DAYS = [1, 5, 20]
LAG_SOURCES_CANDIDATE = [
    "y_known_at_t",
    "trailing_vol__annual_decimel_5d_calculated",
    "trailing_vol__annual_decimel_20d_calculated",
    "trailing_vol_annual_decimel_5d_calculated",
    "trailing_vol_annual_decimel_20d_calculated",
    "VIX",
]
LAG_SOURCES = [c for c in LAG_SOURCES_CANDIDATE if c in df_feat.columns]

lag_col_names = []
for col in LAG_SOURCES:
    for L in LAG_DAYS:
        lag_name = f"{col}_lag{L}"
        df_feat[lag_name] = df_feat.groupby(TICKER_COL)[col].shift(L)
        lag_col_names.append(lag_name)

# Final feature list
FEATURE_COLS = list(dict.fromkeys(BASE_FEATURES + CALENDAR_FEATURES + lag_col_names))

print("Base features used:", BASE_FEATURES)
print("Lag sources used:", LAG_SOURCES)
print("Total features:", len(FEATURE_COLS))
print("First 15 features:", FEATURE_COLS[:15])


Base features used: ['y_known_at_t', 'trailing_vol_annual_decimel_5d_factset', 'trailing_vol_annual_decimel_20d_factset', 'trailing_vol_annual_decimel_5d_calculated', 'trailing_vol_annual_decimel_20d_calculated', 'VIX', 'NYGOLDS', 'OIL_WTI_S', 'US_10Y_BOND_YLD', 'US_3M_TB_YLD']
Lag sources used: ['y_known_at_t', 'trailing_vol_annual_decimel_5d_calculated', 'trailing_vol_annual_decimel_20d_calculated', 'VIX']
Total features: 25
First 15 features: ['y_known_at_t', 'trailing_vol_annual_decimel_5d_factset', 'trailing_vol_annual_decimel_20d_factset', 'trailing_vol_annual_decimel_5d_calculated', 'trailing_vol_annual_decimel_20d_calculated', 'VIX', 'NYGOLDS', 'OIL_WTI_S', 'US_10Y_BOND_YLD', 'US_3M_TB_YLD', 'day_of_week', 'month', 'quarter', 'y_known_at_t_lag1', 'y_known_at_t_lag5']


## Modeling helpers

- `compute_metrics`: RMSE, MAE, R-squared
- `fit_xgb_with_early_stopping`: handles XGBoost version differences for early stopping


In [7]:
# Purpose:
#   Define evaluation metrics and an XGBoost fit helper with early stopping.

def compute_metrics(y_true, y_pred, label: str = "") -> dict:
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)

    rmse = float(math.sqrt(mean_squared_error(y_true, y_pred)))
    mae  = float(mean_absolute_error(y_true, y_pred))
    r2   = float(r2_score(y_true, y_pred))

    return {"Model": label, "RMSE": rmse, "MAE": mae, "R2": r2}


def fit_xgb_with_early_stopping(X_fit, y_fit, X_eval, y_eval, params: dict) -> XGBRegressor:
    """Fit an XGBRegressor with early stopping in a version-safe way."""

    fit_sig = inspect.signature(XGBRegressor.fit).parameters

    # Some versions accept early_stopping_rounds in fit()
    if "early_stopping_rounds" in fit_sig:
        model = XGBRegressor(**params)
        model.fit(
            X_fit, y_fit,
            eval_set=[(X_eval, y_eval)],
            early_stopping_rounds=EARLY_STOPPING_ROUNDS,
            verbose=False,
        )
        return model

    # Other versions use constructor argument early_stopping_rounds
    model = XGBRegressor(**params, early_stopping_rounds=EARLY_STOPPING_ROUNDS)
    model.fit(
        X_fit, y_fit,
        eval_set=[(X_eval, y_eval)],
        verbose=False,
    )
    return model


## Walk-forward engine (per ticker) — full coverage after warm-up

We want `baseline_xgb_walkforward` for **every row/date after warm-up**, without gaps caused by calendar folds.

### Warm-up definition (per ticker)
We start predicting at the earliest row index `start_pred_idx` such that:
- `start_pred_idx >= MAX_LAG_DAYS` (so lag features exist), and
- the number of **non-null target** rows *before* `start_pred_idx` is at least `MIN_TRAIN_ROWS_Y`

### Prediction schedule (no fold gaps)
After `start_pred_idx`, we predict in contiguous blocks of `WF_STEP_ROWS` rows:

1. Train on all rows strictly before the block start (using only rows where the target is non-null)
2. Predict the next block (we predict even if the target is missing; those rows are just not scorable)
3. Move to the next block immediately (contiguous)

This produces baseline values everywhere after warm-up.


In [8]:
# Purpose:
#   Walk-forward expanding-window predictions per ticker, with FULL coverage after warm-up.
#
# Key behavior vs. calendar-year folds:
#   - Predictions start immediately after warm-up (not “Jan 1 of some year”)
#   - Predictions proceed in contiguous blocks of WF_STEP_ROWS (no fold-boundary gaps)
#   - We predict rows even if TARGET_COL is NaN (baseline column still filled, but those rows cannot be scored)

def _start_index_after_warmup(
    g: pd.DataFrame,
    target_col: str,
    max_lag_days: int,
    min_train_rows_y: int,
) -> int | None:
    """Return earliest position i such that:
       - i >= max_lag_days
       - count of non-null target rows in g[0:i] >= min_train_rows_y
       If never satisfied, return None.
    """
    y_nonnull_cum = g[target_col].notna().astype(int).cumsum()
    eligible_before_i = y_nonnull_cum.shift(1).fillna(0).astype(int)
    cond = (eligible_before_i >= min_train_rows_y) & (np.arange(len(g)) >= max_lag_days)
    idx = np.where(cond.values)[0]
    return int(idx[0]) if len(idx) else None


def run_walk_forward_xgb_full_coverage_per_ticker(
    df_full: pd.DataFrame,
    tickers: list[str],
    feature_cols: list[str],
    target_col: str,
    date_col: str,
    ticker_col: str,
    max_lag_days: int,
    min_train_rows_y: int,
    step_rows: int,
    early_stop_split: float,
    xgb_params: dict | None = None,
) -> tuple[pd.Series, pd.DataFrame]:
    """Return (predictions, diagnostics).

    predictions:
      Series aligned to df_full.index named 'baseline_xgb_walkforward'
      NaN only during warm-up (or if a ticker never reaches warm-up thresholds).

    diagnostics:
      Block-by-block info to explain coverage and performance.
    """

    if xgb_params is None:
        xgb_params = XGB_PARAMS

    preds = pd.Series(np.nan, index=df_full.index, name="baseline_xgb_walkforward")
    diag_rows = []

    for ticker in tickers:
        g = df_full[df_full[ticker_col] == ticker].sort_values(date_col).copy()
        if g.empty:
            continue

        start_idx = _start_index_after_warmup(
            g=g,
            target_col=target_col,
            max_lag_days=max_lag_days,
            min_train_rows_y=min_train_rows_y,
        )

        if start_idx is None:
            print(f"\n=== {ticker}: SKIPPED (never reaches warm-up thresholds) ===")
            diag_rows.append({
                "ticker": ticker,
                "status": "SKIPPED_NEVER_WARMED_UP",
                "rows": int(len(g)),
                "first_date": str(g[date_col].min().date()),
                "last_date": str(g[date_col].max().date()),
            })
            continue

        print(f"\n=== {ticker}: rows={len(g)}, start_pred_idx={start_idx}, start_date={g.iloc[start_idx][date_col].date()} ===")

        # Predict in contiguous blocks starting immediately after warm-up
        for block_start in range(start_idx, len(g), step_rows):
            block_end = min(block_start + step_rows, len(g))

            train = g.iloc[:block_start].copy()
            pred  = g.iloc[block_start:block_end].copy()

            # Hard no-leakage guard: train dates strictly before prediction dates
            if not train.empty and train[date_col].max() >= pred[date_col].min():
                raise AssertionError("Time leakage guard failed: train max date >= pred min date")

            # Train only on rows where target exists
            train_y = train.dropna(subset=[target_col]).copy()
            if len(train_y) < min_train_rows_y:
                # Should not happen after start_idx, but keep safe.
                continue

            # Chronological split for early stopping (still before pred block)
            split_ix = int(len(train_y) * early_stop_split)
            split_ix = max(1, min(split_ix, len(train_y) - 1))

            fit_part  = train_y.iloc[:split_ix]
            eval_part = train_y.iloc[split_ix:]

            X_fit  = fit_part[feature_cols]
            y_fit  = fit_part[target_col]
            X_eval = eval_part[feature_cols]
            y_eval = eval_part[target_col]

            X_pred = pred[feature_cols]

            model = fit_xgb_with_early_stopping(X_fit, y_fit, X_eval, y_eval, params=xgb_params)

            y_hat = model.predict(X_pred)
            y_hat = np.clip(y_hat, 0.0, None)  # volatility is non-negative

            preds.loc[pred.index] = y_hat

            # Score only where target exists in the prediction block
            pred_scored = pred.dropna(subset=[target_col]).copy()
            if len(pred_scored) > 0:
                y_true = pred_scored[target_col].astype(float).values
                y_pred = preds.loc[pred_scored.index].astype(float).values
                rmse = float(math.sqrt(mean_squared_error(y_true, y_pred)))
                mae  = float(mean_absolute_error(y_true, y_pred))
                r2   = float(r2_score(y_true, y_pred))
            else:
                rmse, mae, r2 = np.nan, np.nan, np.nan

            best_iter = getattr(model, "best_iteration", None)

            diag_rows.append({
                "ticker": ticker,
                "status": "OK",
                "block_start_date": str(pred[date_col].min().date()),
                "block_end_date": str(pred[date_col].max().date()),
                "train_end_date": str(train[date_col].max().date()) if not train.empty else None,
                "train_rows_target_nonnull": int(len(train_y)),
                "pred_rows": int(len(pred)),
                "pred_rows_target_nonnull": int(len(pred_scored)),
                "rmse": rmse,
                "mae": mae,
                "r2": r2,
                "best_iteration": int(best_iter) if best_iter is not None else np.nan,
            })

        coverage = float(preds.loc[g.index].notna().mean())
        print(f"Ticker baseline coverage: {coverage:.1%}")

    diag_df = pd.DataFrame(diag_rows)
    print(f"\nOverall baseline non-null rate: {float(preds.notna().mean()):.1%}")
    return preds, diag_df


print("Walk-forward full-coverage engine defined: run_walk_forward_xgb_full_coverage_per_ticker()")


Walk-forward full-coverage engine defined: run_walk_forward_xgb_full_coverage_per_ticker()


In [9]:
# Purpose:
#   Run the walk-forward FULL-COVERAGE engine for either one selected ticker or all tickers,
#   then attach the baseline predictions to the dataset.

print("Starting walk-forward XGBoost baseline (full coverage after warm-up)...")

wf_predictions, diag_df = run_walk_forward_xgb_full_coverage_per_ticker(
    df_full=df_feat,
    tickers=tickers_to_run,
    feature_cols=FEATURE_COLS,
    target_col=TARGET_COL,
    date_col=DATE_COL,
    ticker_col=TICKER_COL,
    max_lag_days=MAX_LAG_DAYS,
    min_train_rows_y=MIN_TRAIN_ROWS_Y,
    step_rows=WF_STEP_ROWS,
    early_stop_split=WF_EARLY_STOP_SPLIT,
    xgb_params=XGB_PARAMS,
)

df_out = df_feat.copy()
df_out["baseline_xgb_walkforward"] = wf_predictions.values

print("\nDiagnostics (first 20 rows):")
display(diag_df.head(20))


Starting walk-forward XGBoost baseline (full coverage after warm-up)...

=== BEDZ: rows=1181, start_pred_idx=125, start_date=2021-10-18 ===
Ticker baseline coverage: 89.4%

=== FTXG: rows=2333, start_pred_idx=125, start_date=2017-03-22 ===
Ticker baseline coverage: 94.6%

=== IYZ: rows=2767, start_pred_idx=125, start_date=2015-07-01 ===
Ticker baseline coverage: 95.5%

=== KBE: rows=2767, start_pred_idx=125, start_date=2015-07-01 ===
Ticker baseline coverage: 95.5%

=== KCE: rows=2767, start_pred_idx=125, start_date=2015-07-01 ===
Ticker baseline coverage: 95.5%

=== KIE: rows=2767, start_pred_idx=125, start_date=2015-07-01 ===
Ticker baseline coverage: 95.5%

=== KRE: rows=2767, start_pred_idx=125, start_date=2015-07-01 ===
Ticker baseline coverage: 95.5%

=== PEJ: rows=2767, start_pred_idx=125, start_date=2015-07-01 ===
Ticker baseline coverage: 95.5%

=== PNQI: rows=2767, start_pred_idx=125, start_date=2015-07-01 ===
Ticker baseline coverage: 95.5%

=== PXQ: rows=2767, start_pred_id

,ticker,status,block_start_date,block_end_date,train_end_date,train_rows_target_nonnull,pred_rows,pred_rows_target_nonnull,rmse,mae,r2,best_iteration
0,BEDZ,OK,2021-10-18,2022-10-17,2021-10-15,125,252,252,0.168694,0.132336,-0.509169,19
1,BEDZ,OK,2022-10-18,2023-10-18,2022-10-17,377,252,252,0.125305,0.111026,-1.898945,2
2,BEDZ,OK,2023-10-19,2024-10-18,2023-10-18,629,252,252,0.106074,0.090142,-1.227839,69
3,BEDZ,OK,2024-10-21,2025-10-22,2024-10-18,881,252,252,0.137422,0.085640,0.164328,48
4,BEDZ,OK,2025-10-23,2025-12-31,2025-10-22,1133,48,43,0.068660,0.052528,0.185864,69
5,FTXG,OK,2017-03-22,2018-03-21,2017-03-21,125,252,252,0.063131,0.043300,0.174697,17
6,FTXG,OK,2018-03-22,2019-03-22,2018-03-21,377,252,252,0.070953,0.052223,-0.259992,39
7,FTXG,OK,2019-03-25,2020-03-23,2019-03-22,629,252,252,0.188174,0.095829,0.053247,9
8,FTXG,OK,2020-03-24,2021-03-23,2020-03-23,881,252,252,0.104639,0.084328,-0.129865,80
9,FTXG,OK,2021-03-24,2022-03-22,2021-03-23,1133,252,252,0.082901,0.054883,-0.003258,7


## Evaluation vs rolling baselines (optional but recommended)

We compare only on rows where:

- the target is non-null
- `baseline_xgb_walkforward` is non-null
- all four rolling baselines are non-null

This produces an apples-to-apples comparison.


In [10]:
# Purpose:
#   Compare walk-forward XGBoost to the rolling baselines on the same rows.

eval_cols = ROLLING_BASELINE_COLS + ["baseline_xgb_walkforward", TARGET_COL]
eval_cols = [c for c in eval_cols if c in df_out.columns]

wf_eval_df = df_out.dropna(subset=eval_cols).copy()
print(f"Evaluation rows: {len(wf_eval_df):,}")

results = []
for col in ROLLING_BASELINE_COLS:
    if col in wf_eval_df.columns:
        results.append(compute_metrics(wf_eval_df[TARGET_COL], wf_eval_df[col], label=col))

results.append(compute_metrics(wf_eval_df[TARGET_COL], wf_eval_df["baseline_xgb_walkforward"], label="baseline_xgb_walkforward"))

results_df = pd.DataFrame(results).set_index("Model").sort_values("RMSE")
display(results_df.round(6))


Evaluation rows: 91,845


,RMSE,MAE,R2
Model,,,
baseline_avg_20,0.132548,0.082769,0.275253
baseline_avg_5,0.134684,0.088165,0.251697
baseline_xgb_walkforward,0.141088,0.087989,0.178846
baseline_ols_20,0.166909,0.109371,-0.149223
baseline_ols_5,0.228210,0.154950,-1.148376


## Save outputs to Google Drive

Single ETF mode:
- Writes `vol_dataset_<TICKER>_with_xgb_walkforward_baseline_fullcoverage.csv` (if `OUTPUT_ONLY_SELECTED_TICKER=True`)

All ETF mode:
- Writes `vol_dataset_with_xgb_walkforward_baseline_fullcoverage.csv`

Also writes:
- `xgb_walkforward_fullcoverage_diagnostics.csv` (block-by-block diagnostics)


In [11]:
# Purpose:
#   Save outputs to Google Drive.

# Decide output dataframe scope
if SELECTED_TICKER and OUTPUT_ONLY_SELECTED_TICKER:
    save_df = df_out[df_out[TICKER_COL] == SELECTED_TICKER].copy()
    out_name = f"vol_dataset_{SELECTED_TICKER}_with_xgb_walkforward_baseline_fullcoverage.csv"
else:
    save_df = df_out.copy()
    out_name = "vol_dataset_with_xgb_walkforward_baseline_fullcoverage.csv"

# Primary output: dataset + baseline
out_path_csv = os.path.join(DRIVE_FOLDER, out_name)
save_df.to_csv(out_path_csv, index=False)
print("Saved primary output:", out_path_csv)

# Diagnostics (block-by-block)
out_path_diag = os.path.join(DRIVE_FOLDER, "xgb_walkforward_fullcoverage_diagnostics.csv")
diag_df.to_csv(out_path_diag, index=False)
print("Saved diagnostics:", out_path_diag)

print("\nDone.")


Saved primary output: /content/drive/MyDrive/Colab Notebooks/vol_dataset_with_xgb_walkforward_baseline_fullcoverage.csv
Saved diagnostics: /content/drive/MyDrive/Colab Notebooks/xgb_walkforward_fullcoverage_diagnostics.csv

Done.


## Sanity checks (fast)

What you should verify:

1. **No negative predictions** (we clip at zero).
2. **Coverage behaves as intended**:
   - Baseline is missing mainly at the beginning (warm-up).
   - After the first prediction for a ticker, there should be **no NaN gaps** for that ticker.


In [12]:
# Purpose:
#   Quick sanity checks for the walk-forward baseline.

pred = df_out["baseline_xgb_walkforward"]

neg = int((pred.dropna() < 0).sum())
print(f"Negative predictions: {neg}  (should be 0)")

mn = float(pred.min())
mx = float(pred.max())
mu = float(pred.mean())
print(f"Prediction range: min={mn:.6f}, max={mx:.6f}, mean={mu:.6f}")

# NaN coverage by year (helps confirm only early years are missing)
tmp = df_out[[DATE_COL, TICKER_COL, "baseline_xgb_walkforward"]].copy()
tmp["year"] = tmp[DATE_COL].dt.year
nan_rate_by_year = tmp.groupby("year")["baseline_xgb_walkforward"].apply(lambda s: float(s.isna().mean()))
display(nan_rate_by_year.to_frame("nan_rate").round(4))

# Gap check: after the first non-null baseline for a ticker, there should be no further NaNs.
def has_gap_after_first_prediction(s: pd.Series) -> bool:
    s_nonnull = s.dropna()
    if len(s_nonnull) == 0:
        return True
    first_idx = s_nonnull.index[0]
    return bool(s.loc[first_idx:].isna().any())

gap_report = df_out.groupby(TICKER_COL)["baseline_xgb_walkforward"].apply(has_gap_after_first_prediction)
print("\nGap check (after first prediction):")
display(gap_report.value_counts().to_frame("num_tickers"))

# Show tickers that still have gaps (if any)
tickers_with_gaps = gap_report[gap_report].index.tolist()
if tickers_with_gaps:
    print("Tickers with gaps:", tickers_with_gaps[:25], ("..." if len(tickers_with_gaps) > 25 else ""))


Negative predictions: 0  (should be 0)
Prediction range: min=0.051273, max=0.848076, mean=0.203253


,nan_rate
year,
2014,1.0000
2015,0.4959
2016,0.0163
2017,0.0063
2018,0.0144
2019,0.0000
2020,0.0000
2021,0.0139
2022,0.0000



Gap check (after first prediction):


,num_tickers
baseline_xgb_walkforward,
False,36
